In [1]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter, AutoMinorLocator)
from matplotlib.ticker import LogLocator, LogFormatter
from matplotlib.ticker import LogLocator, NullFormatter
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
from constants import *
from functions import *

In [2]:
eV = 1.602176e-12
mphi = 30.

In [3]:
scenarios = ["BD+43"]
shocks = ['RS']

In [4]:
# Dictionary where data will be stored
distributions = {}

for sc in scenarios:
    distributions[sc] = {}
    for shock in shocks:

        distributions[sc][shock] = {}

        file = f"../../../output/diste_{sc}_{shock}.dat"
        df = pd.read_csv(file, delim_whitespace=True, header=None)
        
        file_total = f"../../../output/diste_RS_{sc}_total.dat"
        df_tot = pd.read_csv(file_total, delim_whitespace=True, header=None)
        
        file_unique = f"../../../output/diste_RS_{sc}_unique_total.dat"
        df_unique = pd.read_csv(file_unique, delim_whitespace=True, header=None)

        distributions[sc][shock]["angle"] = df.iloc[:, 0] * 180.0 / np.pi
        distributions[sc][shock]["E_e"]   = df.iloc[:, 1]
        distributions[sc][shock]["diste"] = df.iloc[:, 2] * 4. * mphi
        distributions[sc][shock]["total"] = df_tot.iloc[:,1] * 4.
        distributions[sc][shock]["unique"] = df_unique.iloc[:,1] * 4.
        

FileNotFoundError: [Errno 2] No such file or directory: '../../../output/diste_RS_BD+43_unique_total.dat'

In [ ]:
fig, ax = plt.subplots(
    figsize=(0.6*text_width, 0.6*2./3.*text_width)
)
cmap = cm.plasma

d = distributions[sc][shock]
theta = d["angle"]
norm = mcolors.Normalize(vmin=min(theta), vmax=max(theta))

# Distributions per angle
for cn in np.unique(theta):
    val = (d["E_e"] * eV)**2 * d["diste"]
    mask = (theta == cn) & (val > 0)
    idx = np.where(mask)[0]
    if len(idx) > 1:
        ax.plot(
            np.log10(d["E_e"])[idx[1:]],
            np.log10(val[idx[1:]]),
            color=cmap(norm(cn)),
            linewidth=1.
        )
        
# Total curve
ax.plot(
    np.log10(d["E_e"]),
    np.log10((d["E_e"] * eV)**2 *  d["unique"]),
    color="olivedrab",
    alpha=1.,
    linewidth=3
)

ax.plot(
    np.log10(d["E_e"]),
    np.log10((d["E_e"] * eV)**2 *  d["total"]),
    color="black",
    alpha=1.,
    linewidth=3
)

# =========================
# Ticks and axes
# =========================
ax.annotate(
    r'$p_1 = -1$',
    xy=(0.05, 0.65),
    xycoords="axes fraction",
    ha="left", va="top",
)

ax.annotate(
    r'$E_{\rm break} \sim 2$ GeV',
    xy=(0.4, 0.87),
    xycoords="axes fraction",
    ha="center", va="bottom",
)

ax.annotate(
    r'$p_2 = 3$',
    xy=(0.7, 0.5),
    xycoords="axes fraction",
    ha="right", va="top",
)

ax.set_xlim(6, 15)
ax.set_xticks([9, 12])
ax.set_xticks([7, 8, 10, 11, 13, 14], minor=True)
ax.tick_params(
    axis="both", which="both",
    top=True, bottom=True,
    left=True, right=True,
    direction="in"
)

ax.set_ylim(36.5, 47.5)
ax.set_yticks([38, 40, 42, 44, 46])
ax.set_yticks([37, 39, 41, 43, 45, 47], minor=True)

ax.xaxis.set_minor_formatter(NullFormatter())
ax.yaxis.set_minor_formatter(NullFormatter())

ax.xaxis.set_ticks_position('both')
ax.yaxis.set_ticks_position('both')

# =========================
# Colour bar
# =========================
pos = ax.get_position()
cax = fig.add_axes([
    pos.x1 + 0.02,
    pos.y0,
    0.03,
    pos.height
])
sm = plt.cm.ScalarMappable(
    cmap=cmap,
    norm=mcolors.Normalize(vmin=0, vmax=120)
)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_ticks([0, 30, 60, 90, 120])
cbar.set_label(r'$\theta\,[^\circ]$')

# =========================
# Global labels
# =========================
ax.set_xlabel(r'$\log_{10}\,(E_e \, [{\rm eV}])$')
ax.set_ylabel(r'$\log_{10}\,(E_e^2 N_e(E_e))$ [erg]')

plt.show()

In [ ]:
#fig.savefig('diste_BD+43.pdf', bbox_inches='tight', pad_inches = 0.01)